In [2]:
import logging
import time
from functools import wraps

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    filename="api.log"
)

logger = logging.getLogger("API")

console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(levelname)s | %(message)s")
console.setFormatter(formatter)
logger.addHandler(console)


def log_timing(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        logger.info(f"{func.__name__} executed in {end-start:.4f} seconds")
        return result
    return wrapper


def log_io(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger.debug(f"INPUT → args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        logger.debug(f"OUTPUT → {result}")
        return result
    return wrapper


def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(user_role, *args, **kwargs):
            if user_role != role:
                logger.error(f"ACCESS DENIED for role '{user_role}' on {func.__name__}")
                raise PermissionError("Unauthorized access")
            logger.info(f"Access granted for role '{user_role}'")
            return func(user_role, *args, **kwargs)
        return wrapper
    return decorator


@log_timing
@log_io
@require_role("admin")
def delete_user(user_role, user_id):
    return f"User {user_id} deleted successfully"


delete_user("admin", 101)

try:
    delete_user("guest", 202)
except PermissionError:
    pass

INFO | Access granted for role 'admin'
INFO | delete_user executed in 0.0142 seconds
ERROR | ACCESS DENIED for role 'guest' on delete_user
